# Week 5: Is It Any Good? Testing a Model

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: The memorizing student

A student memorizes every answer on the practice test and scores 100 percent. On the real test, with new questions, they score 40 percent. Did they learn anything? A model can do the same thing. Today we learn to catch it.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

penguins = load("penguins.csv").dropna()
features = ["bill_length_mm", "flipper_length_mm"]
X = penguins[features]
y = penguins["species"]

## Teach 1: Train on some, test on the rest

We hide part of the data from the model (the **test set**), train on the rest (the **training set**), then check how well it does on the hidden rows. That is the only honest score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)
print("Training rows:", len(X_train), " Test rows:", len(X_test))

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

test_predictions = knn.predict(X_test)
print("Accuracy on hidden penguins:", round(accuracy_score(y_test, test_predictions) * 100, 1), "%")

## Teach 2: Where does it go wrong?

Accuracy is one number. A **confusion matrix** shows exactly which species got mixed up with which. Rows are the truth, columns are the prediction. Numbers on the diagonal are correct.

**Concept checkpoint:** which two species do you expect to be confused most often? (Look back at last week's chart.)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Train with `n_neighbors=1` and with `n_neighbors=25`. Print the test accuracy for each. Which is better?

### Medium
Loop k from 1 to 30, record the test accuracy for each, and plot accuracy vs k. Where is the sweet spot?

### Spicy
**Biased training data.** Train only on penguins from the islands Biscoe and Dream, then test on penguins from Torgersen. What happens to accuracy and why? Write two sentences connecting this to real world AI.

In [ ]:
# MILD: two values of k
for k in [1, 25]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print("k =", k, " accuracy =", round(acc * 100, 1), "%")

In [ ]:
# MEDIUM: accuracy vs k
ks = list(range(1, 31))
scores = []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.plot(ks, scores, marker="o", color="#2271b1")
plt.title("Test accuracy for different k")
plt.xlabel("k (number of neighbors)")
plt.ylabel("Accuracy")
plt.show()
print("Best k:", ks[scores.index(max(scores))])

In [ ]:
# SPICY: biased training data
train_rows = penguins[penguins["island"].isin(["Biscoe", "Dream"])]
test_rows  = penguins[penguins["island"] == "Torgersen"]
print("Species living on Torgersen:", test_rows["species"].unique())

biased = KNeighborsClassifier(n_neighbors=5).fit(train_rows[features], train_rows["species"])
preds = biased.predict(test_rows[features])
print("Accuracy on Torgersen:", round(accuracy_score(test_rows["species"], preds) * 100, 1), "%")
print(pd.crosstab(test_rows["species"], preds))
# Torgersen only has Adelie penguins, and Adelie also live on the training islands,
# so accuracy may still be good. Now try training on Biscoe only and testing on Dream:
train_b = penguins[penguins["island"] == "Biscoe"]
test_d  = penguins[penguins["island"] == "Dream"]
m = KNeighborsClassifier(n_neighbors=5).fit(train_b[features], train_b["species"])
print("Train Biscoe, test Dream accuracy:",
      round(accuracy_score(test_d["species"], m.predict(test_d[features])) * 100, 1), "%")
print("Chinstrap penguins never appear on Biscoe, so the model has never seen one.")

## Extra activities (if you finish early)

- Change `random_state` in `train_test_split` to a different number. Does accuracy change? Why does that matter?
- Compute accuracy on the **training** rows. Is it higher than on the test rows? Which number would you report to a customer?
- Find a news story about an AI that worked in testing but failed in the real world.

## Reflection

- Why do we hide some data from the model?
- What does a confusion matrix tell you that accuracy does not?